In [0]:
import sys
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/ScoringPhysics")
sys.path.append("/Workspace/Users/karthik.raj@bio-techne.com/Scfv_Toolkit")

In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import os
path_boltz = os.path.abspath('/Workspace/Users/karthik.raj@bio-techne.com/boltz')
os.chdir(path_boltz)

%pip install --no-dependencies .

In [0]:
import sys
import os
import shutil
import tarfile
from pathlib import Path

# --- CONFIGURATION ---
# Permanent Storage (Unity Catalog Volume)
VOLUME_DIR = Path("/Volumes/sandbox/model_weights/protein_hunter")
# Fast Local Storage (Ephemeral SSD)
LOCAL_DIR = Path.home() / ".boltz"

# Create directories
os.makedirs(VOLUME_DIR, exist_ok=True)
if os.path.exists(LOCAL_DIR):
    # Clean slate for local dir to avoid half-extracted messes
    if os.path.islink(LOCAL_DIR):
        os.remove(LOCAL_DIR)
    else:
        shutil.rmtree(LOCAL_DIR)
os.makedirs(LOCAL_DIR, exist_ok=True)

print(f"📍 Hybrid Setup Initiated")
print(f"   Storage: {VOLUME_DIR}")
print(f"   Execution: {LOCAL_DIR}")

# --- HELPER FUNCTIONS ---
def install_from_volume():
    print("🔄 Found files in Volume! Linking to local SSD...")
    
    # 1. Link the Weights (Instant)
    # We don't copy the 3GB file, we just symlink it.
    vol_ckpt = VOLUME_DIR / "boltz2_conf.ckpt"
    local_ckpt = LOCAL_DIR / "boltz2_conf.ckpt"
    if vol_ckpt.exists():
        if os.path.exists(local_ckpt): os.remove(local_ckpt)
        os.symlink(vol_ckpt, local_ckpt)
        print("   ✅ Weights linked.")
    
    # 2. Extract the Molecules (Fast on SSD)
    # We copy the tarball and extract it locally
    vol_tar = VOLUME_DIR / "mols.tar"
    local_tar = LOCAL_DIR / "mols.tar"
    
    if vol_tar.exists():
        print("   ⏳ Copying CCD tarball to local SSD...")
        shutil.copy(vol_tar, local_tar)
        
        print("   📦 Extracting molecules locally (this will be fast)...")
        with tarfile.open(local_tar) as tar:
            tar.extractall(path=LOCAL_DIR)
        print("   ✅ CCD Data ready.")

def download_and_backup():
    print("⬇️ Files missing in Volume. Downloading fresh...")
    
    # Setup path to import the downloader
    repo_path = os.getcwd()
    sys.path.insert(0, os.path.join(repo_path, "boltz_ph"))
    try:
        from boltz.main import download_boltz2
    except ImportError:
        sys.path.insert(0, os.path.join(repo_path, "boltz_ph", "src"))
        from boltz.main import download_boltz2

    # Download to LOCAL SSD first (Fastest)
    download_boltz2(LOCAL_DIR)
    
    print("💾 Backing up to Volume for future runs...")
    # Copy the big files to the volume
    local_ckpt = LOCAL_DIR / "boltz2_conf.ckpt"
    local_tar = LOCAL_DIR / "mols.tar" # The downloader usually keeps the tar?
    
    # Note: boltz downloader extracts immediately. We might need to re-tar or just check what's there.
    # Usually it leaves boltz2_conf.ckpt and mols/ folder.
    
    # Backup Weights
    if local_ckpt.exists():
        shutil.copy(local_ckpt, VOLUME_DIR / "boltz2_conf.ckpt")
        print("   ✅ Weights backed up.")
        
    # Backup Molecules (Re-tar them for efficient storage)
    if (LOCAL_DIR / "mols").exists():
        print("   zzZ Compressing molecules for backup...")
        with tarfile.open(VOLUME_DIR / "mols.tar", "w") as tar:
            tar.add(LOCAL_DIR / "mols", arcname="mols")
        print("   ✅ CCD Data backed up.")

# --- EXECUTION LOGIC ---
# Check if the Volume already has the goods
if (VOLUME_DIR / "boltz2_conf.ckpt").exists() and (VOLUME_DIR / "mols.tar").exists():
    install_from_volume()
else:
    download_and_backup()

print("\n🚀 Ready to run! Environment is optimized for Databricks.")

In [0]:
dbutils.widgets.text('design_csv_name', 'rfd_af2_passed_chunk_0.csv')
design_csv_name = dbutils.widgets.get('design_csv_name')
dbutils.widgets.text('path_design_campaign', "/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular_trial3")
path_design_campaign = dbutils.widgets.get('path_design_campaign')
print("Design_CSV_Input: ", design_csv_name)
print("Path_Design_Campaign: ", path_design_campaign)

In [0]:
import pandas as pd
path_csv_folder = f"{path_design_campaign}/rfdiffusion"
design_path = path_csv_folder + f"/rfd_af2_passed_boltz_input/{design_csv_name}"
df_designs = pd.read_csv(design_path, index_col = 0)
df_designs

In [0]:
def compute_binder_rmsd(rfd_input_path, boltz_design_path):
    """ Extract Binder Coordinates -> Align on Binder Coordinates -> Compute RMSD on Binder Coordinates"""
    atom_array_design = extract_atom_array(boltz_design_path, ca_only = False)
    binder_coords_design = atom_array_design[atom_array_design.chain_id == 'A'].coord
    atom_array_input = extract_atom_array(rfd_input_path, ca_only = False)
    binder_coords_input = atom_array_input[atom_array_input.chain_id == 'B'].coord
    aligned_design_binder_coords, transformation = struc.superimpose(binder_coords_input, binder_coords_design)
    rmsd = struc.rmsd(aligned_design_binder_coords, binder_coords_input)
    return rmsd

In [0]:
from StrucTools import *
from RunBoltz import *

volume_save_path = f"{path_csv_folder}/boltz_holo_structures/"
#df_sample = df_designs.sample(n = 2, random_state= 42)
all_design_metrics = []
for index, row in df_designs.iterrows():
    design_name = row['design_name']

    # Boltz Inputs
    seq_binder = row['seq_binder']
    seq_target = row['seq_target']
    seq_list = [seq_binder, seq_target]
    msa_options = ['empty', '']
    template_paths = [''] * len(seq_list)
    entity_type = ['protein'] * len(seq_list)
    num_models = 5
    ligand = '[Mg+2]'
    
    # Contact Check Inputs: Epitope Residues (1-indexed) or empty list
    desired_epitope_residues = [12,13,14,15,74,75,76,77,78,79,80,115,116,117]

    df_design_metrics = boltz_predict_analyze(design_name=design_name, volume_save_path=volume_save_path, seq_list=seq_list,msa_options=msa_options,template_paths=template_paths, entity_type=entity_type, desired_epitope_residues=desired_epitope_residues, num_models = num_models, ligand = ligand)

    # Iterate through all the models predicted by Boltz and compute binder RMSD wrt to RFD Holo structure
    path_input_pdb = row['design_pdb_path']
    list_holo_rfd_boltz_binder_rmsd = []
    df_design_metrics['rmsd_holo_binder_rfd_boltz'] = df_design_metrics['path_structure'].apply(lambda x: compute_binder_rmsd(rfd_input_path= path_input_pdb, boltz_design_path= x))

    
    all_design_metrics.append(df_design_metrics)

df_all_design_metrics = pd.concat(all_design_metrics).reset_index(drop=True)
df_all_design_metrics




In [0]:
# Save CSV of results to folder
df_all_design_metrics.to_csv(f"{path_csv_folder}/rfd_af2_passed_boltz_input/boltz_holo_{design_csv_name}", index=False)

### Run as a one-off case

In [0]:
cleaved_ectodomain_seq = "TDENRCLKANAKSCGECIQAGPNCGWCTNSTFLQEGMPTSARCDDLEALKKKGCPPDDENPRGSKDIKKNKNVTNRSKGTAEKLKPEDITQIQPQQLVLRLRSGEPQTFTLKFKRAEDYPIDLYYLMDLSYSMKDDLENVKSLGTDLMNEMRRITSDFRIGFGSFVESYKNVLSLTNKGEVFNELVGKQRISGNLDSPEGGFDAIMQVAVCGSLIGWRNVTRLLVFSTDAGFHFAGDGKLGGIVLPNDGQCHLENNMYTMSHYYDYPSIAHLVQKLSENNIQTIFAVTEEFQPVYKELKNLIPKSAVGTLSANSSNVIQLIIDAYNSLS"
epitope_of_interest = "KNVLSLTNKGEVFNELVGKQRISGN"
epitope_start = cleaved_ectodomain_seq.find(epitope_of_interest)
desired_epitope_residues = list(range(epitope_start + 1, epitope_start + len(epitope_of_interest) + 1))
desired_epitope_residues

In [0]:
from StrucTools import *
from RunBoltz import *
volume_save_path = "/Workspace/Users/karthik.raj@bio-techne.com/MotifScaffold/RFDiffusion/Double_Strand_Globular/designs_cif/"
design_name = "ts2_integrin_B1"
# Boltz Inputs
seq_vh = "DVKLVESGGGLVKPGGSLKLSCAASGFTFSSYTMSWVRQTPEKRLEWVATISSGGSYTYYPDSVKGRFTISRDKAKNTLYLQMGSLKSEDTAMYYCTRIGYDEDYAMDHWGQGTSVTVSS"
seq_linker = "GGGGSGGGGSGGGGS"
seq_vl = "EIVVTQSPTTMAASPGDKITITCSVSSIISSNYLHWYSQKPGFSPKLLIYRTSNLASGVPPRFSGSGSGTSYSLTIGTMEAEDVATYYCQQGSDIPLTFGDGTKLDLK"
seq_binder = seq_vh + seq_linker + seq_vl
seq_target = cleaved_ectodomain_seq
seq_list = [seq_binder, seq_target]
msa_options = ['', '']
template_paths = [''] * len(seq_list)
entity_type = ['protein'] * len(seq_list)
num_models = 5
ligand = ''

# Contact Check Inputs: Epitope Residues (1-indexed) or empty list
desired_epitope_residues = desired_epitope_residues

df_design_metrics = boltz_predict_analyze(design_name=design_name, volume_save_path=volume_save_path, seq_list=seq_list,msa_options=msa_options,template_paths=template_paths, entity_type=entity_type, desired_epitope_residues=desired_epitope_residues, num_models = num_models, ligand = ligand)

df_design_metrics

In [0]:
for structure_path in df_design_metrics['path_structure'].iloc[:5]:
    visualize_structure(structure_path)

In [0]:
import os
os.listdir("/Volumes/sandbox/karthik/motif_scaffolding/alpha2_beta1_double_globular/rfdiffusion/rfd_af2_passed_boltz_input/")